# Tool Use (Function Calling)

The **Tool Use** pattern enables an LLM to interact with external APIs, databases, and services by deciding autonomously when and how to call specific functions. It bridges the gap between the LLM's reasoning capabilities and real-world execution.

The process follows a well-defined loop:
1. **Tool Definition** — Describe available tools to the LLM (name, purpose, parameter schema).
2. **LLM Decision** — The model receives the user query and decides if a tool call is necessary.
3. **Structured Call Generation** — The model outputs a structured object specifying the tool name and arguments.
4. **Tool Execution** — Your code executes the actual function with those arguments.
5. **Result Injection** — The tool result is returned to the model as context.
6. **Final Response** — The model produces a final answer incorporating the tool's output.

This loop repeats until the model produces a response without requesting any further tool calls.

**Use cases:** financial data retrieval, inventory queries, real-time search, code execution, sending notifications, controlling external systems.

## Implementation with Flyte v2

This notebook implements the tool-use agent loop using **Flyte v2 primitives** and the **Anthropic tool-use API**. Each tool call is a traced checkpoint; the full tool-call history is stored as a typed dataclass so Flyte can display, retry, and pass it between tasks.

#### LangChain vs Flyte v2 — Key Differences

| Aspect | LangChain (`AgentExecutor`) | Flyte v2 |
|--------|----------------------------|----------|
| **Tool definition** | `@langchain_tool` decorator | Plain typed Python functions + JSON schema dict |
| **Agent loop** | `AgentExecutor.ainvoke` (black box) | Explicit `while` loop — every step is visible |
| **Checkpointing** | None | `@flyte.trace` per LLM call and per tool call |
| **State** | In-process `AgentScratchpad` | `AgentState` dataclass — serializable, inspectable |
| **Observability** | Verbose logging | Structured outputs in Flyte UI + live report |
| **Execution** | In-process only | Local or remote (containers on Kubernetes) |

### 1. Install dependencies

In [ ]:
!uv pip install flyte anthropic -U

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [3]:
!flyte start devbox

flyte-devbox
  Waiting for flyte cluster to be ready ━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:02:115m 83% 0:02:11
╭──────────────────────────────── Flyte Devbox ────────────────────────────────╮
│ Flyte devbox cluster is ready!                                               │
│                                                                              │
│   🚀 UI:             ]8;id=4839095;http://localhost:30080/v2\http://localhost:30080/v2]8;;\                               │
│   🐳 Image Registry: localhost:30000                                         │
╰──────────────────────────────────────────────────────────────────────────────╯


### 2. Export your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...


### 3. Import dependencies and configure the Flyte TaskEnvironment

In [30]:
import json
import os
from dataclasses import dataclass, field
from datetime import timedelta
from typing import Any

from anthropic import AsyncAnthropic
import flyte
import flyte.report

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="tool-use-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.25.0")
)

tool_env = flyte.TaskEnvironment(
    name="tool_use_env",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the data models

`AgentState` captures the full tool-call history as a typed, serializable dataclass. This makes the agent's reasoning process:
- **Inspectable** — every tool call and result is visible in the Flyte UI
- **Durable** — state survives pod restarts via Flyte's object storage
- **Composable** — downstream tasks can consume the full history for auditing or further processing

In [31]:
@dataclass
class ToolCall:
    """A single tool invocation requested by the LLM."""
    tool_use_id: str = ""
    name: str = ""
    inputs: dict = None  # type: ignore[assignment]

    def __post_init__(self) -> None:
        if self.inputs is None:
            self.inputs = {}


@dataclass
class ToolResult:
    """The result of executing a tool call."""
    tool_use_id: str = ""
    name: str = ""
    output: str = ""
    is_error: bool = False


@dataclass
class AgentState:
    """Full history of an agent's tool-use session."""
    query: str = ""
    tool_calls: list[ToolCall] = None  # type: ignore[assignment]
    tool_results: list[ToolResult] = None  # type: ignore[assignment]
    final_answer: str = ""
    turns: int = 0

    def __post_init__(self) -> None:
        if self.tool_calls is None:
            self.tool_calls = []
        if self.tool_results is None:
            self.tool_results = []

### 5. Define the tools

Tools are plain Python functions — no special decorator needed, but here we create a custom `tool` decorator that auto-generates the Anthropic schema from type hints and
docstrings. 
This explicit schema approach means:
- The LLM understands exactly what each tool does and what parameters it expects
- Tool execution is transparent — you control the dispatch logic
- New tools can be added without any framework changes

This example builds a **financial research assistant** that can look up stock prices and calculate portfolio metrics.

In [32]:
import inspect
from typing import Any, TypedDict, get_type_hints, get_origin, get_args

TOOL_REGISTRY: dict[str, callable] = {}
TOOL_SCHEMAS: dict[str, dict] = {}  # keyed by name — prevents duplicates on re-import


def _hint_to_schema(hint) -> dict:
    if hint is str:   return {"type": "string"}
    if hint is int:   return {"type": "integer"}
    if hint is float: return {"type": "number"}
    if hint is bool:  return {"type": "boolean"}
    origin = get_origin(hint)
    if origin is list:
        args = get_args(hint)
        return {"type": "array", "items": _hint_to_schema(args[0]) if args else {"type": "object"}}
    if isinstance(hint, type) and issubclass(hint, dict) and hasattr(hint, "__annotations__"):
        sub = get_type_hints(hint)
        return {
            "type": "object",
            "properties": {k: _hint_to_schema(v) for k, v in sub.items()},
            "required": list(sub.keys()),
        }
    return {"type": "object"}


def _parse_args_docstring(doc: str) -> dict[str, str]:
    param_docs: dict[str, str] = {}
    in_args = False
    for line in doc.split("\n"):
        stripped = line.strip()
        if stripped.lower() in ("args:", "arguments:", "parameters:"):
            in_args = True
            continue
        if in_args:
            if stripped and not line.startswith(" ") and stripped.endswith(":"):
                break
            if ":" in stripped:
                name, desc = stripped.split(":", 1)
                param_docs[name.strip()] = desc.strip()
    return param_docs


def tool(fn):
    """Registers a function as an agent tool. Safe to re-run — overwrites by name."""
    doc = inspect.getdoc(fn) or fn.__name__
    param_docs = _parse_args_docstring(doc)
    hints = {k: v for k, v in get_type_hints(fn).items() if k != "return"}
    sig = inspect.signature(fn)

    properties: dict[str, dict] = {}
    required: list[str] = []
    for name, param in sig.parameters.items():
        schema = _hint_to_schema(hints.get(name, Any))
        if name in param_docs:
            schema["description"] = param_docs[name]
        if param.default is not inspect.Parameter.empty:
            schema["default"] = param.default
        else:
            required.append(name)
        properties[name] = schema

    TOOL_SCHEMAS[fn.__name__] = {
        "name": fn.__name__,
        "description": doc.split("\n")[0],
        "input_schema": {"type": "object", "properties": properties, "required": required},
    }
    TOOL_REGISTRY[fn.__name__] = fn
    return fn


def get_tool_schemas() -> list[dict]:
    """Return the current tool schemas as a list for the Anthropic API."""
    return list(TOOL_SCHEMAS.values())

We now use the decorator to define the tools that will be available to our agent:

In [33]:
class Holding(TypedDict):
    ticker: str
    shares: float


@tool
def get_stock_price(ticker: str) -> dict:
    """Get the current price, daily change %, and market cap for a stock ticker.

    Args:
        ticker: The stock ticker symbol, e.g. 'AAPL' for Apple.
    """
    simulated = {
        "AAPL": {"price": 178.15, "change_pct": 1.23, "market_cap_billions": 2780},
        "MSFT": {"price": 425.50, "change_pct": -0.45, "market_cap_billions": 3160},
        "GOOGL": {"price": 175.30, "change_pct": 0.87, "market_cap_billions": 2190},
        "NVDA": {"price": 875.40, "change_pct": 3.12, "market_cap_billions": 2150},
    }
    ticker = ticker.upper()
    if ticker not in simulated:
        return {"error": f"Ticker '{ticker}' not found. Available: {list(simulated.keys())}"}
    return {"ticker": ticker, **simulated[ticker]}


@tool
def calculate_portfolio_value(holdings: list[Holding]) -> dict:
    """Calculate the total current market value of a stock portfolio.

    Args:
        holdings: List of positions, each with a ticker symbol and number of shares.
    """
    prices = {"AAPL": 178.15, "MSFT": 425.50, "GOOGL": 175.30, "NVDA": 875.40}
    positions, total = [], 0.0
    for h in holdings:
        ticker = h["ticker"].upper()
        price = prices.get(ticker)
        if price is None:
            return {"error": f"Unknown ticker: {ticker}"}
        value = price * h["shares"]
        total += value
        positions.append({"ticker": ticker, "shares": h["shares"], "value": round(value, 2)})
    largest = max(positions, key=lambda p: p["value"])
    return {"total_value": round(total, 2), "positions": positions, "largest_position": largest}


@tool
def search_financial_news(query: str, max_results: int = 3) -> dict:
    """Search for recent financial news articles matching the query.

    Args:
        query: Search query string for financial news.
        max_results: Maximum number of articles to return (1-10).
    """
    articles = [
        {"title": "Apple Reports Record Q4 Revenue of $94.9B",
         "summary": "Apple exceeded analyst expectations with strong iPhone and Services growth.",
         "date": "2024-11-01"},
        {"title": "Microsoft Cloud Revenue Surges 33% on AI Demand",
         "summary": "Azure growth accelerated as enterprise AI adoption drives cloud spending.",
         "date": "2024-10-30"},
        {"title": "NVIDIA Data Center Revenue Hits $30B in Q3",
         "summary": "GPU demand from AI hyperscalers continues to outpace supply.",
         "date": "2024-11-20"},
        {"title": "Alphabet Beats Estimates with 15% Revenue Growth",
         "summary": "Google Search and YouTube both showed strong advertising recovery.",
         "date": "2024-10-29"},
    ]
    query_lower = query.lower()
    matched = [a for a in articles if any(
        w in a["title"].lower() or w in a["summary"].lower()
        for w in query_lower.split()
    )]
    return {"articles": (matched or articles)[:max_results], "total_found": len(matched or articles)}

### 6. Define the traced tool executor and agent loop

The `@flyte.trace` decorator on `_execute_tool` makes each tool call a named checkpoint in the Flyte UI. If the pod crashes mid-loop, Flyte retries from the last completed checkpoint rather than restarting from scratch — saving all previously completed tool calls.

The agent loop is an explicit `while` loop, giving you full control and visibility:
- Each turn is logged to `AgentState`
- Tool calls and results are fully typed
- The stopping condition (`stop_reason == "end_turn"`) is explicit

In [34]:
SYSTEM_PROMPT = """\
You are a financial research assistant with access to real-time stock data and news.
Use the available tools to retrieve accurate data before answering.
Always cite the specific data you retrieved when making claims.
Be concise and factual. If a tool returns an error, explain the limitation clearly."""


@flyte.trace
async def _execute_tool(tool_call: ToolCall) -> ToolResult:
    """Traced — each tool call appears as a named checkpoint in the Flyte UI."""
    fn = TOOL_REGISTRY.get(tool_call.name)
    if fn is None:
        return ToolResult(
            tool_use_id=tool_call.tool_use_id,
            name=tool_call.name,
            output=json.dumps({"error": f"Unknown tool: {tool_call.name}"}),
            is_error=True,
        )
    try:
        result = fn(**tool_call.inputs)
        return ToolResult(
            tool_use_id=tool_call.tool_use_id,
            name=tool_call.name,
            output=json.dumps(result),
        )
    except Exception as exc:
        return ToolResult(
            tool_use_id=tool_call.tool_use_id,
            name=tool_call.name,
            output=json.dumps({"error": str(exc)}),
            is_error=True,
        )


async def _call_llm(messages: list[dict], tools: list[dict]):
    """Plain async helper — not traced, so Flyte never tries to serialize messages."""
    client = AsyncAnthropic()
    return await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4096,
        tools=tools,
        system=SYSTEM_PROMPT,
        messages=messages,
    )

### 7. Define the agent loop task

The task runs the tool-use loop: call the LLM, execute any requested tools, inject results, repeat until the model produces a final answer (`stop_reason == "end_turn"`). A maximum turn guard prevents runaway loops.

In [35]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _content_to_dicts(content_blocks) -> list[dict]:
    """Convert Anthropic SDK content blocks to plain dicts for serialization."""
    result = []
    for block in content_blocks:
        if block.type == "text":
            result.append({"type": "text", "text": block.text})
        elif block.type == "tool_use":
            result.append({"type": "tool_use", "id": block.id, "name": block.name, "input": block.input})
    return result


@tool_env.task(
    retries=3,
    timeout=timedelta(minutes=15),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def tool_use_agent(query: str, max_turns: int = 10) -> AgentState:
    """
    Financial research agent with tool use, Flyte v2 native.

    The agent loop:
      1. Send query + available tool schemas to the LLM.
      2. If stop_reason == 'tool_use': extract tool_use blocks, execute them, inject results.
      3. If stop_reason == 'end_turn': extract final text answer and exit.
      4. Repeat up to max_turns to prevent infinite loops.
    """
    import json as _json

    tools = list({s["name"]: s for s in get_tool_schemas()}.values())

    state = AgentState(query=query)
    messages: list[dict] = [{"role": "user", "content": query}]
    report_html_sections: list[str] = []

    for turn in range(max_turns):
        state.turns = turn + 1
        response = await _call_llm(messages, tools)

        # ── Final answer ──────────────────────────────────────────────────────
        if response.stop_reason == "end_turn":
            state.final_answer = next(
                (b.text for b in response.content if b.type == "text"), ""
            )
            report_html_sections.append(
                f"<section><h2>Final Answer (turn {turn + 1})</h2>"
                f"<pre style='background:#e8f5e9;padding:1em;border-radius:4px'>"
                f"{_html_escape(state.final_answer)}</pre></section>"
            )
            await flyte.report.replace.aio(
                "<html><body style='font-family:sans-serif;max-width:900px;margin:auto;padding:1.5em'>"
                "<h1>Tool Use Agent — Execution Report</h1>"
                + "".join(report_html_sections)
                + "</body></html>"
            )
            await flyte.report.flush.aio()
            break

        # ── Tool calls requested ──────────────────────────────────────────────
        if response.stop_reason == "tool_use":
            # Convert SDK objects to plain dicts before appending — Flyte traces
            # checkpoint `messages` and cannot serialize TextBlock/ToolUseBlock.
            messages.append({"role": "assistant", "content": _content_to_dicts(response.content)})

            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                tool_call = ToolCall(
                    tool_use_id=block.id,
                    name=block.name,
                    inputs=block.input,
                )
                state.tool_calls.append(tool_call)

                result = await _execute_tool(tool_call)
                state.tool_results.append(result)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result.output,
                })

                result_data = _json.loads(result.output)
                report_html_sections.append(
                    f"<section><h2>Turn {turn + 1} — Tool: <code>{block.name}</code></h2>"
                    f"<h3>Inputs</h3><pre style='background:#e3f2fd;padding:.8em;border-radius:4px'>"
                    f"{_html_escape(_json.dumps(block.input, indent=2))}</pre>"
                    f"<h3>Result</h3><pre style='background:#f3e5f5;padding:.8em;border-radius:4px'>"
                    f"{_html_escape(_json.dumps(result_data, indent=2))}</pre></section><hr/>"
                )

            messages.append({"role": "user", "content": tool_results})

            await flyte.report.replace.aio(
                "<html><body style='font-family:sans-serif;max-width:900px;margin:auto;padding:1.5em'>"
                "<h1>Tool Use Agent — Execution Report</h1>"
                + "".join(report_html_sections)
                + "</body></html>"
            )
            await flyte.report.flush.aio()

    return state

### 8. Run locally

In [36]:
run = flyte.run(
    tool_use_agent,
    query="What is Apple's current stock price and how has it been performing recently?",
    max_turns=10,
)
run.wait()
print(run.outputs()[0])
print(run.url)


> Building 1 image...

> Building image tool-use-agent for environment tool_use_env

✓ Built image for environment tool_use_env: localhost:30000/tool-use-agent:b7e21ad4642010db89f88189de935c8b

Output()

AgentState(query="What is Apple's current stock price and how has it been performing recently?", tool_results=[{'tool_use_id': 'toolu_017jTyoeVHov1k59Ezj1nTWj', 'name': 'get_stock_price', 'output': '{"ticker": "AAPL", "price": 178.15, "change_pct": 1.23, "market_cap_billions": 2780}', 'is_error': False}, {'tool_use_id': 'toolu_01KgumGjqeohYVRT9MAyn83c', 'name': 'search_financial_news', 'output': '{"articles": [{"title": "Apple Reports Record Q4 Revenue of $94.9B", "summary": "Apple exceeded analyst expectations with strong iPhone and Services growth.", "date": "2024-11-01"}], "total_found": 1}', 'is_error': False}], tool_calls=[{'tool_use_id': 'toolu_017jTyoeVHov1k59Ezj1nTWj', 'name': 'get_stock_price', 'inputs': {'ticker': 'AAPL'}}, {'tool_use_id': 'toolu_01KgumGjqeohYVRT9MAyn83c', 'name': 'search_financial_news', 'inputs': {'query': 'Apple AAPL stock performance'}}], final_answer="Here's a snapshot of how Apple (**AAPL**) is currently doing:\n\n---\n\n### 📈 Current Stock Data\n- **Pr

### Running remotely

When running on a Flyte cluster, each tool call and LLM turn appears as a named checkpoint in the UI. The live report tab shows tool inputs and outputs as they happen.

1. Create the secret on the cluster:

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

2. Switch to remote execution:

In [ ]:
run = flyte.run(
    tool_use_agent,
    query="Find recent NVIDIA news and calculate my portfolio value: 10 NVDA, 20 AAPL, 15 MSFT.",
    max_turns=10,
)
run.wait()

if run.error():
    print(f"Task failed: {run.error()}")
else:
    state = run.outputs()[0]
    print(state.final_answer)


## Scaling the pattern

### Parallel tool execution

When the LLM requests multiple tools in a single turn, execute them concurrently with `asyncio.gather` instead of sequentially. This can dramatically reduce latency when tools are I/O-bound (e.g., HTTP calls to external APIs).

In [ ]:
import asyncio

# Replace the sequential loop in tool_use_agent with parallel execution:
async def execute_tools_parallel(tool_use_blocks: list) -> list[ToolResult]:
    """
    Execute all tool calls in a single LLM turn concurrently.

    When the LLM requests get_stock_price('AAPL') AND search_financial_news('Apple')
    in the same turn, both can run simultaneously rather than one after the other.
    For N tools with average latency L, this reduces tool execution time from N*L to L.
    """
    coroutines = [
        _execute_tool(ToolCall(
            tool_use_id=block.id,
            name=block.name,
            inputs=block.input,  # already a dict in Anthropic's API
        ))
        for block in tool_use_blocks
        if block.type == "tool_use"
    ]
    return await asyncio.gather(*coroutines)

### Adding real tools

To connect real APIs, replace the simulated functions with actual HTTP calls. The agent loop requires no changes — only the tool implementations change:

In [ ]:
# Example: replacing get_stock_price with a real Alpha Vantage call
# (requires ALPHA_VANTAGE_API_KEY secret)
import httpx

async def get_stock_price_real(ticker: str) -> dict:
    """Production implementation using Alpha Vantage API."""
    api_key = os.environ["ALPHA_VANTAGE_API_KEY"]
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={ticker}&apikey={api_key}"
    async with httpx.AsyncClient() as client:
        resp = await client.get(url, timeout=10.0)
        resp.raise_for_status()
        data = resp.json().get("Global Quote", {})
        return {
            "ticker": ticker,
            "price": float(data.get("05. price", 0)),
            "change_pct": float(data.get("10. change percent", "0%").strip("%")),
        }

## Scaling the pattern

The devbox runs each task in a fresh container. For production workloads with many short LLM calls, `ReusePolicy` eliminates cold-start overhead by keeping a pool of warm containers ready.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Requires a Union deployment — not supported on the local devbox
from datetime import timedelta

production_tool_use = flyte.TaskEnvironment(
    name="tool_use_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)